# מפגש 15 — מחברת הדגמה: KNN ביד בלבד
עבור שאילתה חדשה ללא תווית אמת נחשב מרחקים, נדרג שכנים ונצביע; אין Accuracy ואין מודל Scikit-learn. נתוני A–E הם דוגמת המורה ממקור ההשתלמות.

## צעד 1: נוסחת פיתגורס ביד
בין (0,0) ל־(3,4), ריבועי הפרשים 9 ו־16, השורש של 25 הוא 5. הסבירו מהו מרחק ולאן שייכת תווית המחלקה.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
distance_34=math.sqrt((3-0)**2+(4-0)**2)
distance_12=math.sqrt((4-1)**2+(6-2)**2)
print('Distance (0,0) to (3,4):',distance_34)
print('Distance (1,2) to (4,6):',distance_12)
assert distance_34==5
assert distance_12==5

## צעד 2: חמש תצפיות מתויגות, ושאילתה חדשה
המזהה id והתווית label אינם חלק מנוסחת המרחק; f1,f2 הן שתי התכונות שעל אותו סרגל הוראתי.

In [ ]:
training_df=pd.DataFrame({
'id':['A','B','C','D','E'],
'f1':[3,5,5,7,3],
'f2':[2,3,5,3,8],
'label':['Red','Blue','Blue','Red','Red']})
query_f1,query_f2=3,3
original=training_df.copy(deep=True)
display(training_df)
assert training_df.shape==(5,4)
assert [query_f1,query_f2]==[3,3]
assert training_df['label'].tolist()==['Red','Blue','Blue','Red','Red']

## צעד 3: נקודות במישור עם יחס צירים שווה
על המפה צבע מסמן תווית ידועה, והכוכב את השאילתה ללא תווית. חזו מי קרוב לפי המפה, ואז אמתו מספרית.

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
for label,color in [('Red','firebrick'),('Blue','royalblue')]:
    part=training_df[training_df['label']==label]
    ax.scatter(part['f1'],part['f2'],s=95,label=label,color=color)
for i in range(len(training_df)):
    row=training_df.iloc[i]
    ax.annotate(row['id'],(row['f1']+.1,row['f2']+.1))
ax.scatter([query_f1],[query_f2],marker='*',s=230,color='black',label='Unknown query')
ax.set_xlabel('f1 (teaching scale)')
ax.set_ylabel('f2 (teaching scale)')
ax.set_title('Training points and query; no true query label')
ax.set_aspect('equal',adjustable='box')
ax.legend()
plt.tight_layout()
plt.show()

## צעד 4: הפרשים, ריבועי הפרשים, שורש ומיון
לפי נוסחת דוגמת המקור מתקבלים מרחקים 1,2,√8,4,5. ניצור עותק כדי לא לזהם את נתוני האימון במאפיין distance חדש.

In [ ]:
distance_df=training_df.copy(deep=True)
distance_df['dx']=distance_df['f1']-query_f1
distance_df['dy']=distance_df['f2']-query_f2
distance_df['squared']=distance_df['dx']**2+distance_df['dy']**2
distance_df['distance']=distance_df['squared']**0.5
ranked_df=distance_df.sort_values('distance',kind='stable')
display(ranked_df[['id','dx','dy','squared','distance','label']])
assert ranked_df['id'].tolist()==['A','B','C','D','E']
assert np.allclose(ranked_df['distance'],[1,2,np.sqrt(8),4,5])
assert ranked_df['squared'].tolist()==[1,4,8,16,25]
assert training_df.equals(original)
assert 'distance' not in training_df.columns

## צעד 5: K=1,3,5 — שלוש הצבעות, אותה שאילתה
כל שכן מקבל קול אחד. mode עשויה להחזיר יותר ממחלקה במקרה של תיקו — אין להציג מצב כזה כניצחון יחיד.

In [ ]:
expected={1:'Red',3:'Blue',5:'Red'}
for k in [1,3,5]:
    neighbors=ranked_df.head(k)
    modes=neighbors['label'].mode().tolist()
    counts=neighbors['label'].value_counts().to_dict()
    print('K=',k,'neighbors=',neighbors['id'].tolist(),'votes=',counts,'mode=',modes)
    assert modes==[expected[k]]
assert len(ranked_df)==5

## צעד 6: מחברים את הנקודה החדשה לשלושת השכנים הקרובים
קווים לשכנים A,B,C. הגרף מציג את המרחב החזותי אבל הטבלה המספרית קובעת את הדירוג.

In [ ]:
fig,ax=plt.subplots(figsize=(6,5))
for label,color in [('Red','firebrick'),('Blue','royalblue')]:
    part=training_df[training_df['label']==label]
    ax.scatter(part['f1'],part['f2'],s=95,label=label,color=color)
for i in range(len(training_df)):
    row=training_df.iloc[i]
    ax.annotate(row['id'],(row['f1']+.1,row['f2']+.1))
for i in range(3):
    row=ranked_df.iloc[i]
    ax.plot([query_f1,row['f1']],[query_f2,row['f2']],color='gray',linestyle='--',alpha=.7)
ax.scatter([query_f1],[query_f2],marker='*',s=230,color='black',label='Unknown query')
ax.set_xlabel('f1 (teaching scale)')
ax.set_ylabel('f2 (teaching scale)')
ax.set_title('Three nearest neighbors: A, B, C')
ax.set_aspect('equal',adjustable='box')
ax.legend()
plt.tight_layout()
plt.show()

## צעד 7: מקרה גבול — תיקו בשלוש מחלקות
האם K אי־זוגי מבטיח רוב אם קיימות A,B,C? לא. כאן mode מחזירה שלוש תוויות; מדיניות הכרעה דורשת החלטה חיצונית מראש.

In [ ]:
three_labels=pd.Series(['A','B','C'])
tie=three_labels.mode().tolist()
print('Tie modes:',tie)
assert tie==['A','B','C']
assert len(tie)==3
assert 7>len(training_df)
assert training_df.equals(original)

## הגנה קצרה
מהו X ומהו y? האם כוכב השאילתה תורם לקבוצת האימון? מה השתנה בין K=1,3,5? האם זו הצלחה מאומתת? לא: אין תווית אמת לשאילתה. מה הסכנה במרחקים על מחיר ₪ ו־RAM GB בלי הכנה על Train?